# Baseline Model Training

# Import Dependencies

In [1]:
import joblib
import numpy as np
import pandas as pd

from abc import ABC, abstractmethod


from xgboost import XGBClassifier
from sklearn import metrics
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression


# Load Dataset

In [2]:
df = pd.read_csv("../data/cleaned_data.csv")
df

,accession,length,gc_content,sequence,label
0,NC_036615.1,2330,40.26,GGCATAAGCAGAGGATTTTATAACAATGGAAATAAACCCATATCTA...,influenza D
1,NC_036616.1,2364,39.89,GGCATAAGCAGAGGATGTCACTACTATTAACGCTCGCAAAAGAGTA...,influenza D
2,NC_036617.1,1775,44.00,GGCATAAGCAGGAGATTATTAAGCAATATGGACTCAACAAAAGCCC...,influenza D
3,NC_036618.1,2049,43.34,AGCATAAGCAGGAGATTTTCAAAGATGTTTTTGCTTCTAGCAACAA...,influenza D
4,NC_036619.1,2195,39.54,GGCATAAGCAGGAGATTTAGAAATGTCTAGTGTAATCAGAGAAATC...,influenza D
...,...,...,...,...,...
10462,AF389122.1,890,44.72,AGCAAAAGCAGGGTGACAAAGACATAATGGATCCAAACACTGTGTC...,H1N1
10463,AF251391.1,988,47.87,TATTGAAAGATGAGCCTTCTAACCGAGGTCGAAACGTATGTTCTCT...,H3N2
10464,AF102656.1,1350,44.30,ATGAATCCAAATCAGAAGATAATAACCATTGGATCAATCTGCATGG...,H5N1
10465,J02150.1,890,44.27,AGCAAAAGCAGGGTGACAAAGACATAATGGATCCAAACACTGTGTC...,H1N1


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10467 entries, 0 to 10466
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   accession   10467 non-null  str    
 1   length      10467 non-null  int64  
 2   gc_content  10467 non-null  float64
 3   sequence    10467 non-null  str    
 4   label       10467 non-null  str    
dtypes: float64(1), int64(1), str(3)
memory usage: 17.0 MB


# Preprocess

In [4]:
vectorizer = CountVectorizer(analyzer="char", ngram_range=(3, 3))
kmer_matrix = vectorizer.fit_transform(df["sequence"])
kmer_names = vectorizer.get_feature_names_out()
kmer_df = pd.DataFrame(kmer_matrix.toarray(), columns=kmer_names, index=df.index)
kmer_freq_df = kmer_df.div(df["length"], axis=0)
kmer_freq_df["label"] = df["label"]
kmer_freq_df

,aaa,aac,aag,aak,aam,aan,aar,aas,aat,aaw,...,ytm,ytr,ytt,yty,ywc,ywg,yyc,yyg,yym,label
0,0.054077,0.025751,0.032618,0.0,0.0,0.0,0.0,0.0,0.032618,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,influenza D
1,0.053723,0.016074,0.040186,0.0,0.0,0.0,0.0,0.0,0.028342,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,influenza D
2,0.043944,0.026479,0.036620,0.0,0.0,0.0,0.0,0.0,0.023662,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,influenza D
3,0.035627,0.019522,0.030747,0.0,0.0,0.0,0.0,0.0,0.022938,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,influenza D
4,0.048747,0.019134,0.038269,0.0,0.0,0.0,0.0,0.0,0.033257,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,influenza D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10462,0.032584,0.017978,0.030337,0.0,0.0,0.0,0.0,0.0,0.023596,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,H1N1
10463,0.025304,0.016194,0.020243,0.0,0.0,0.0,0.0,0.0,0.019231,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,H3N2
10464,0.020741,0.017778,0.016296,0.0,0.0,0.0,0.0,0.0,0.032593,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,H5N1
10465,0.035955,0.017978,0.029213,0.0,0.0,0.0,0.0,0.0,0.024719,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,H1N1


In [5]:
X = kmer_freq_df[kmer_freq_df.columns.drop('label')]
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(kmer_freq_df['label'])
print(X.shape)
print(y.shape)

(10467, 584)
(10467,)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2, stratify=y)

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(8373, 584)
(8373,)
(2094, 584)
(2094,)


# Setup Training

## Universal Model Template

In [7]:
class UniversalModel(ABC):
    def __init__(self, name):
        self.name = name
        self.model = None

    @abstractmethod
    def train(self, X_train, y_train, **kwargs):
        pass

    @abstractmethod
    def predict(self, X):
        pass

    def evaluate(self, X_test, y_test, target_names):
        predictions = self.predict(X_test)

        print(metrics.classification_report(y_test, predictions, target_names=target_names))
        macro_f1 = metrics.f1_score(y_test, predictions, average="macro")
        confusion_matrix = metrics.confusion_matrix(y_test, predictions)
        print(f"{macro_f1=}")
        return macro_f1, confusion_matrix

## Scikit Learn Model

In [8]:
class SklearnModel(UniversalModel):
    def __init__(self, name, sklearn_estimator):
        super().__init__(name)
        self.model = sklearn_estimator

    def train(self, X_train, y_train, **kwargs):
        self.model.fit(X_train, y_train)
        print(f"{self.name} training complete.")

    def predict(self, X):
        return self.model.predict(X)

# Train Models

In [10]:
target_names = [str(cls) for cls in label_encoder.classes_]
print(target_names)
lr_model = SklearnModel("Logistic Regression", LogisticRegression(solver='lbfgs'))
random_forest = SklearnModel("Random Forest", RandomForestClassifier())
xgb_classifier = SklearnModel("XGBClassifier", XGBClassifier())

['H1N1', 'H1N2', 'H3N2', 'H3N8', 'H4N6', 'H5N1', 'H5N2', 'H5N6', 'H5N8', 'H6N2', 'H7N2', 'H7N3', 'H7N7', 'H7N9', 'H9N2', 'influenza B', 'influenza D']


In [11]:
lr_model.train(X_train, y_train)
lr_f1, lr_cm = lr_model.evaluate(X_test, y_test, target_names)

Logistic Regression training complete.
              precision    recall  f1-score   support

        H1N1       0.24      1.00      0.38       494
        H1N2       0.00      0.00      0.00        80
        H3N2       0.00      0.00      0.00       394
        H3N8       0.00      0.00      0.00        56
        H4N6       0.00      0.00      0.00        26
        H5N1       0.00      0.00      0.00       227
        H5N2       0.00      0.00      0.00       156
        H5N6       0.00      0.00      0.00        36
        H5N8       0.00      0.00      0.00        40
        H6N2       0.00      0.00      0.00        20
        H7N2       0.00      0.00      0.00        22
        H7N3       0.00      0.00      0.00       121
        H7N7       0.00      0.00      0.00        32
        H7N9       0.00      0.00      0.00        84
        H9N2       0.00      0.00      0.00       220
 influenza B       0.00      0.00      0.00        56
 influenza D       0.00      0.00      0.0

/home/nifdi/miniconda3/envs/tcgabio/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nifdi/miniconda3/envs/tcgabio/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nifdi/miniconda3/envs/tcgabio/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [12]:
random_forest.train(X_train, y_train)
rf_f1, rf_cm = random_forest.evaluate(X_test, y_test, target_names)

Random Forest training complete.
              precision    recall  f1-score   support

        H1N1       0.91      0.94      0.92       494
        H1N2       0.85      0.51      0.64        80
        H3N2       0.85      0.93      0.89       394
        H3N8       0.83      0.68      0.75        56
        H4N6       0.85      0.65      0.74        26
        H5N1       0.88      0.96      0.92       227
        H5N2       0.97      0.88      0.92       156
        H5N6       0.88      0.83      0.86        36
        H5N8       1.00      0.90      0.95        40
        H6N2       1.00      0.30      0.46        20
        H7N2       0.92      0.55      0.69        22
        H7N3       0.70      0.92      0.79       121
        H7N7       0.71      0.31      0.43        32
        H7N9       0.95      0.74      0.83        84
        H9N2       0.86      0.95      0.90       220
 influenza B       1.00      1.00      1.00        56
 influenza D       1.00      1.00      1.00     

In [13]:
xgb_classifier.train(X_train, y_train)
xgb_f1, xgb_cm = xgb_classifier.evaluate(X_test, y_test, target_names)

XGBClassifier training complete.
              precision    recall  f1-score   support

        H1N1       0.91      0.92      0.91       494
        H1N2       0.72      0.51      0.60        80
        H3N2       0.84      0.92      0.88       394
        H3N8       0.78      0.70      0.74        56
        H4N6       0.95      0.69      0.80        26
        H5N1       0.90      0.96      0.93       227
        H5N2       0.96      0.89      0.92       156
        H5N6       0.91      0.86      0.89        36
        H5N8       1.00      0.93      0.96        40
        H6N2       0.78      0.35      0.48        20
        H7N2       0.92      0.55      0.69        22
        H7N3       0.73      0.87      0.79       121
        H7N7       0.52      0.34      0.42        32
        H7N9       0.88      0.77      0.82        84
        H9N2       0.89      0.93      0.91       220
 influenza B       0.98      0.98      0.98        56
 influenza D       1.00      1.00      1.00     

In [14]:
seeds = np.random.randint(0, 1000, 10)
lr_scores = list()
for seed in seeds:
    print(f"Using Seed {seed}")
    lr_model = SklearnModel("Logistic Regression", LogisticRegression(random_state=seed, solver='lbfgs'))
    lr_model.train(X_train, y_train)
    f1, _ = lr_model.evaluate(X_test, y_test, target_names)
    lr_scores.append(f1)
    print("="*30)
print(f"Average F1 Score over 10 seeds: {np.mean(lr_scores).item()}")

Using Seed 929
Logistic Regression training complete.
              precision    recall  f1-score   support

        H1N1       0.24      1.00      0.38       494
        H1N2       0.00      0.00      0.00        80
        H3N2       0.00      0.00      0.00       394
        H3N8       0.00      0.00      0.00        56
        H4N6       0.00      0.00      0.00        26
        H5N1       0.00      0.00      0.00       227
        H5N2       0.00      0.00      0.00       156
        H5N6       0.00      0.00      0.00        36
        H5N8       0.00      0.00      0.00        40
        H6N2       0.00      0.00      0.00        20
        H7N2       0.00      0.00      0.00        22
        H7N3       0.00      0.00      0.00       121
        H7N7       0.00      0.00      0.00        32
        H7N9       0.00      0.00      0.00        84
        H9N2       0.00      0.00      0.00       220
 influenza B       0.00      0.00      0.00        56
 influenza D       0.00    

/home/nifdi/miniconda3/envs/tcgabio/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nifdi/miniconda3/envs/tcgabio/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nifdi/miniconda3/envs/tcgabio/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

Logistic Regression training complete.
              precision    recall  f1-score   support

        H1N1       0.24      1.00      0.38       494
        H1N2       0.00      0.00      0.00        80
        H3N2       0.00      0.00      0.00       394
        H3N8       0.00      0.00      0.00        56
        H4N6       0.00      0.00      0.00        26
        H5N1       0.00      0.00      0.00       227
        H5N2       0.00      0.00      0.00       156
        H5N6       0.00      0.00      0.00        36
        H5N8       0.00      0.00      0.00        40
        H6N2       0.00      0.00      0.00        20
        H7N2       0.00      0.00      0.00        22
        H7N3       0.00      0.00      0.00       121
        H7N7       0.00      0.00      0.00        32
        H7N9       0.00      0.00      0.00        84
        H9N2       0.00      0.00      0.00       220
 influenza B       0.00      0.00      0.00        56
 influenza D       0.00      0.00      0.0

/home/nifdi/miniconda3/envs/tcgabio/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nifdi/miniconda3/envs/tcgabio/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nifdi/miniconda3/envs/tcgabio/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

Logistic Regression training complete.
              precision    recall  f1-score   support

        H1N1       0.24      1.00      0.38       494
        H1N2       0.00      0.00      0.00        80
        H3N2       0.00      0.00      0.00       394
        H3N8       0.00      0.00      0.00        56
        H4N6       0.00      0.00      0.00        26
        H5N1       0.00      0.00      0.00       227
        H5N2       0.00      0.00      0.00       156
        H5N6       0.00      0.00      0.00        36
        H5N8       0.00      0.00      0.00        40
        H6N2       0.00      0.00      0.00        20
        H7N2       0.00      0.00      0.00        22
        H7N3       0.00      0.00      0.00       121
        H7N7       0.00      0.00      0.00        32
        H7N9       0.00      0.00      0.00        84
        H9N2       0.00      0.00      0.00       220
 influenza B       0.00      0.00      0.00        56
 influenza D       0.00      0.00      0.0

...

/home/nifdi/miniconda3/envs/tcgabio/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nifdi/miniconda3/envs/tcgabio/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nifdi/miniconda3/envs/tcgabio/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

Logistic Regression training complete.
              precision    recall  f1-score   support

        H1N1       0.24      1.00      0.38       494
        H1N2       0.00      0.00      0.00        80
        H3N2       0.00      0.00      0.00       394
        H3N8       0.00      0.00      0.00        56
        H4N6       0.00      0.00      0.00        26
        H5N1       0.00      0.00      0.00       227
        H5N2       0.00      0.00      0.00       156
        H5N6       0.00      0.00      0.00        36
        H5N8       0.00      0.00      0.00        40
        H6N2       0.00      0.00      0.00        20
        H7N2       0.00      0.00      0.00        22
        H7N3       0.00      0.00      0.00       121
        H7N7       0.00      0.00      0.00        32
        H7N9       0.00      0.00      0.00        84
        H9N2       0.00      0.00      0.00       220
 influenza B       0.00      0.00      0.00        56
 influenza D       0.00      0.00      0.0

/home/nifdi/miniconda3/envs/tcgabio/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nifdi/miniconda3/envs/tcgabio/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nifdi/miniconda3/envs/tcgabio/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

Logistic Regression training complete.
              precision    recall  f1-score   support

        H1N1       0.24      1.00      0.38       494
        H1N2       0.00      0.00      0.00        80
        H3N2       0.00      0.00      0.00       394
        H3N8       0.00      0.00      0.00        56
        H4N6       0.00      0.00      0.00        26
        H5N1       0.00      0.00      0.00       227
        H5N2       0.00      0.00      0.00       156
        H5N6       0.00      0.00      0.00        36
        H5N8       0.00      0.00      0.00        40
        H6N2       0.00      0.00      0.00        20
        H7N2       0.00      0.00      0.00        22
        H7N3       0.00      0.00      0.00       121
        H7N7       0.00      0.00      0.00        32
        H7N9       0.00      0.00      0.00        84
        H9N2       0.00      0.00      0.00       220
 influenza B       0.00      0.00      0.00        56
 influenza D       0.00      0.00      0.0

/home/nifdi/miniconda3/envs/tcgabio/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nifdi/miniconda3/envs/tcgabio/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nifdi/miniconda3/envs/tcgabio/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [15]:
rf_scores = list()
for seed in seeds:
    print(f"Using Seed {seed}")
    random_forest = SklearnModel("Random Forest", RandomForestClassifier(random_state=seed))
    random_forest.train(X_train, y_train)
    f1, _ = random_forest.evaluate(X_test, y_test, target_names)
    rf_scores.append(f1)
    print("="*30)
print(f"Average F1 Score over 10 seeds: {np.mean(rf_scores).item()}")

Using Seed 929
Random Forest training complete.
              precision    recall  f1-score   support

        H1N1       0.91      0.93      0.92       494
        H1N2       0.84      0.47      0.61        80
        H3N2       0.85      0.92      0.88       394
        H3N8       0.83      0.70      0.76        56
        H4N6       0.95      0.69      0.80        26
        H5N1       0.89      0.95      0.92       227
        H5N2       0.98      0.89      0.93       156
        H5N6       0.86      0.86      0.86        36
        H5N8       1.00      0.90      0.95        40
        H6N2       1.00      0.25      0.40        20
        H7N2       0.92      0.55      0.69        22
        H7N3       0.68      0.94      0.79       121
        H7N7       0.67      0.25      0.36        32
        H7N9       0.95      0.75      0.84        84
        H9N2       0.86      0.94      0.90       220
 influenza B       1.00      1.00      1.00        56
 influenza D       1.00      1.00

In [16]:
xgb_scores = list()
for seed in seeds:
    print(f"Using Seed {seed}")
    xgb_classifier = SklearnModel("XGBClassifier", XGBClassifier(random_state=seed))    
    xgb_classifier.train(X_train, y_train)
    f1, _ = xgb_classifier.evaluate(X_test, y_test, target_names)
    xgb_scores.append(f1)
    print("="*30)
print(f"Average F1 Score over 10 seeds: {np.mean(xgb_scores).item()}")

Using Seed 929
XGBClassifier training complete.
              precision    recall  f1-score   support

        H1N1       0.91      0.92      0.91       494
        H1N2       0.72      0.51      0.60        80
        H3N2       0.84      0.92      0.88       394
        H3N8       0.78      0.70      0.74        56
        H4N6       0.95      0.69      0.80        26
        H5N1       0.90      0.96      0.93       227
        H5N2       0.96      0.89      0.92       156
        H5N6       0.91      0.86      0.89        36
        H5N8       1.00      0.93      0.96        40
        H6N2       0.78      0.35      0.48        20
        H7N2       0.92      0.55      0.69        22
        H7N3       0.73      0.87      0.79       121
        H7N7       0.52      0.34      0.42        32
        H7N9       0.88      0.77      0.82        84
        H9N2       0.89      0.93      0.91       220
 influenza B       0.98      0.98      0.98        56
 influenza D       1.00      1.00

# Results

In [17]:
res = pd.DataFrame({"LR": lr_scores, "RF": rf_scores, "XGB": xgb_scores})
res.describe().T

,count,mean,std,min,25%,50%,75%,max
LR,10.0,0.022457,3.657118e-18,0.022457,0.022457,0.022457,0.022457,0.022457
RF,10.0,0.798128,5.595593e-03,0.790559,0.792914,0.798828,0.802053,0.805583
XGB,10.0,0.807189,1.170278e-16,0.807189,0.807189,0.807189,0.807189,0.807189


# Save Models

In [5]:
joblib.dump(label_encoder, "../models/baseline/label_encoder.pkl")
joblib.dump(vectorizer, "../models/baseline/vectorizer.pkl")
joblib.dump(lr_model, "../models/baseline/logistic_regression.pkl")
joblib.dump(random_forest, "../models/baseline/random_forest.pkl")
joblib.dump(xgb_classifier, "../models/baseline/xgb_classifier.pkl")

['../models/baseline/vectorizer.pkl']